In [14]:
import sys
from pathlib import Path
import pandas
from datetime import datetime, timezone

project_root = Path.cwd().resolve().parents[1]

sys.path.insert(0, str(project_root))

In [15]:
from app.db.mongodb import db

In [16]:
db.users.find()

In [17]:
users_df = pandas.DataFrame(list(db.users.find()))
interactions_df = pandas.DataFrame(list(db.interactions.find()))

In [18]:
users_df.head()

,_id,name,age,favorite_categories,favorite_items
0,u1,Alice,25,"[Electronics, Books]","[p1, p2, p18]"
1,u2,Bob,31,[Sportswear],"[p6, p7]"
2,u3,Charlie,28,[Home],"[p11, p13]"
3,u4,Diana,22,"[Books, Toys]","[p17, p26]"
4,u5,Ethan,35,[Electronics],"[p3, p4]"


In [19]:
interactions_df.head()

,_id,user_id,product_id,action,rating,timestamp
0,6a984b736c5a44d816f25528,u1,p1,purchase,5.0,2026-08-13 16:14:43.350
1,6a984b736c5a44d816f25529,u1,p2,purchase,4.0,2026-08-15 16:14:43.350
2,6a984b736c5a44d816f2552a,u1,p5,view,NaN,2026-08-18 16:14:43.350
3,6a984b736c5a44d816f2552b,u1,p18,purchase,5.0,2026-08-23 16:14:43.350
4,6a984b736c5a44d816f2552c,u1,p16,view,NaN,2026-08-25 16:14:43.350


In [31]:
def action_weight(row):
    if row["action"] == "purchase":
        return 3.0
    if row["action"] == "rate":
        return 1.0 + (row["rating"] - 1) * 0.5   # 1★→1.0 ... 5★→3.0
    return 0.5   # view

def recency_decay(timestamp, half_life_days=14):
    age_days = (datetime.now(timezone.utc) - timestamp).days
    return 0.5 ** (age_days / half_life_days)

result = action_weight(interactions_df.iloc[0])

result

3.0